# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.

%load_ext dotenv
%dotenv 

In [2]:
import dask.dataframe as dd

c:\Users\NEWPC\miniconda3\envs\dsi_participant\lib\site-packages\dask\dataframe\__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
import pandas as pd
import os
import sys
from glob import glob

sys.path.append(os.getenv('PRICE_DATA'))
from utils.logger import get_logger   
_logs = get_logger(__name__) 

In [4]:
import os
from glob import glob

# Write your code below.
PRICE_DATA = os.getenv("PRICE_DATA")
_logs.info(f"Reading price data from: {PRICE_DATA}")
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive=True)
dd_px = dd.read_parquet(parquet_files).set_index("ticker")


2025-09-28 20:53:44,751, 366802870.py, 6, INFO, Reading price data from: ../../05_src/data/prices/


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [5]:
# Write your code below.

_logs.info(" Starting feature engineering for each ticker")

dd_shift = dd_px.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(Close_lag_1 = x['Close'].shift(1),
                       Adj_Close_lag_1 = x['Adj Close'].shift(1))
)

2025-09-28 20:55:00,064, 3115447876.py, 3, INFO,  Starting feature engineering for each ticker
C:\Users\NEWPC\AppData\Local\Temp\ipykernel_19848\3115447876.py:5: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_shift = dd_px.groupby('ticker', group_keys=False).apply(


In [6]:
dd_rets = dd_shift.assign(
    
    hi_lo_range = lambda x: x['High'] - x['Low'],
    Returns = lambda x: x['Close']/x['Close_lag_1'] - 1)

In [7]:
dd_feat = dd_rets.compute()
dd_feat

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,hi_lo_range,Returns
ticker,,,,,,,,,,,,,
AADR,2010-07-21,25.100000,25.100000,24.700001,24.700001,23.343714,42000.0,AADR.csv,2010,NaN,NaN,0.400000,NaN
AADR,2010-07-22,25.420000,25.420000,25.129999,25.260000,23.872967,17500.0,AADR.csv,2010,24.700001,23.343714,0.290001,0.022672
AADR,2010-07-23,25.540001,25.540001,25.080000,25.280001,23.891865,8600.0,AADR.csv,2010,25.260000,23.872967,0.460001,0.000792
AADR,2010-07-26,25.400000,25.400000,25.219999,25.370001,23.976921,18900.0,AADR.csv,2010,25.280001,23.891865,0.180000,0.003560
AADR,2010-07-27,25.250000,25.290001,25.200001,25.290001,23.901318,8200.0,AADR.csv,2010,25.370001,23.976921,0.090000,-0.003153
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,2020-03-26,4.060000,4.530000,3.880000,4.510000,4.510000,1668500.0,ZIXI.csv,2020,4.000000,4.000000,0.650000,0.127500
ZIXI,2020-03-27,4.490000,4.710000,4.100000,4.600000,4.600000,1146800.0,ZIXI.csv,2020,4.510000,4.510000,0.610000,0.019956
ZIXI,2020-03-30,4.830000,4.870000,4.440000,4.640000,4.640000,1212000.0,ZIXI.csv,2020,4.600000,4.600000,0.430000,0.008696


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [8]:
# Write your code below.

_logs.info("Converting Dask DataFrame to pandas and calculating 10-day moving average of returns")

dd_feat = dd_rets.compute()
dd_feat = dd_feat.assign(
    returns_ma_10 = dd_feat['Returns'].rolling(10).mean()
)


2025-09-28 20:56:00,077, 242924680.py, 3, INFO, Converting Dask DataFrame to pandas and calculating 10-day moving average of returns


In [9]:
dd_feat.head(20)


,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,hi_lo_range,Returns,returns_ma_10
ticker,,,,,,,,,,,,,,
AADR,2010-07-21,25.100000,25.100000,24.700001,24.700001,23.343714,42000.0,AADR.csv,2010,NaN,NaN,0.400000,NaN,NaN
AADR,2010-07-22,25.420000,25.420000,25.129999,25.260000,23.872967,17500.0,AADR.csv,2010,24.700001,23.343714,0.290001,0.022672,NaN
AADR,2010-07-23,25.540001,25.540001,25.080000,25.280001,23.891865,8600.0,AADR.csv,2010,25.260000,23.872967,0.460001,0.000792,NaN
AADR,2010-07-26,25.400000,25.400000,25.219999,25.370001,23.976921,18900.0,AADR.csv,2010,25.280001,23.891865,0.180000,0.003560,NaN
AADR,2010-07-27,25.250000,25.290001,25.200001,25.290001,23.901318,8200.0,AADR.csv,2010,25.370001,23.976921,0.090000,-0.003153,NaN
AADR,2010-07-28,25.250000,25.290001,25.120001,25.200001,23.816256,4900.0,AADR.csv,2010,25.290001,23.901318,0.170000,-0.003559,NaN
AADR,2010-07-29,25.299999,25.299999,25.020000,25.020000,23.646139,1200.0,AADR.csv,2010,25.200001,23.816256,0.279999,-0.007143,NaN
AADR,2010-07-30,24.990000,25.100000,24.990000,25.100000,23.721750,600.0,AADR.csv,2010,25.020000,23.646139,0.110001,0.003197,NaN
AADR,2010-08-02,25.700001,25.709999,25.440001,25.620001,24.213196,7000.0,AADR.csv,2010,25.100000,23.721750,0.269999,0.020717,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

The choice depends largely on your goals and the size of yout dataset. If you are working with small to medium-sized dat, converting to pandas is generally fine. However, for large, or extremely large datasets, switching to pandas can be memory-intensive and slow.

In many cases, it is better to stick with Dask, specially when you are dealing with large scale data where computation are likely to be memory-expensive.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.